In [ ]:
#| default_exp handlers.pipeline.output

# Output

Final column guards, metadata merge policy, and NetCDF serialization for the GeneralHandler runtime.

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
from marisco.callbacks import Transformer, RenameColumnsCB, Callback
from marisco.metadata import GlobAttrsFeeder, DepthRangeCB, TimeRangeCB, KeyValuePairCB
from marisco.encoders import NetCDFEncoder
from marisco.configs import NC_VARS
from marisco.handlers.pipeline.contracts import HandlerConfig, _MARIS_REQUIRED, ensure_known_global_attrs

## Output Helpers

In [ ]:
#| export
class _SimpleBboxCB(Callback):
    """Compute geospatial bounding box from LAT/LON via pure pandas min/max.
    Replaces BboxCB (Shapely-free). WKT polygon delegated to DB server."""
    def __call__(self, obj):
        all_df = pd.concat(obj.dfs.values())
        obj.attrs.update({
            'geospatial_lat_min': str(all_df['LAT'].min()),
            'geospatial_lat_max': str(all_df['LAT'].max()),
            'geospatial_lon_min': str(all_df['LON'].min()),
            'geospatial_lon_max': str(all_df['LON'].max()),
        })


def validate_required_columns(tfm: Transformer) -> None:
    "Raise KeyError if any MARIS-required column is missing from an output group."
    for grp, df in tfm.dfs.items():
        missing = _MARIS_REQUIRED - set(df.columns)
        if missing:
            raise KeyError(f"Group '{grp}' missing required MARIS columns: {sorted(missing)}")


def project_netcdf_columns(tfm: Transformer) -> None:
    "Drop non-NetCDF noise columns by projecting each group through NC_VARS."
    for grp in list(tfm.dfs):
        grp_guard = {k: k for k in NC_VARS if k in tfm.dfs[grp].columns}
        Transformer({grp: tfm.dfs[grp]}, cbs=[RenameColumnsCB(grp_guard)], inplace=True)()


def build_global_attrs(tfm: Transformer, cfg: HandlerConfig) -> dict[str, str]:
    "Merge declarative and computed NetCDF global attributes through a shared policy helper."
    computed_attrs = GlobAttrsFeeder(tfm.dfs, cbs=[
        _SimpleBboxCB(),
        DepthRangeCB(),
        TimeRangeCB(),
        KeyValuePairCB('keywords', ', '.join(cfg.keywords)),
        KeyValuePairCB('publisher_postprocess_logs', ', '.join(tfm.logs)),
    ])()
    global_attrs = {**cfg.global_attrs, **computed_attrs}
    return ensure_known_global_attrs(global_attrs)


def write_netcdf(tfm: Transformer, cfg: HandlerConfig) -> None:
    "Encode transformed DataFrames to MARIS NetCDF4; KeyError on missing required MARIS columns."
    validate_required_columns(tfm)
    project_netcdf_columns(tfm)
    global_attrs = build_global_attrs(tfm, cfg)
    NetCDFEncoder(tfm.dfs, dest_fname=cfg.fname_out, global_attrs=global_attrs).encode()